Machine learning CW.  
Housing data, Available at: https://www.zillow.com/research/data/.  
Student ID: 00017747.  

1

In [13]:
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

os.environ.pop("MPLBACKEND", None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16

## 1. Data Loading & Melting

In [14]:
df = pd.read_csv("housing_data.csv")
df_ny = df[df['StateName'] == 'NY'].copy()

id_vars = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
date_cols = [c for c in df_ny.columns if c not in id_vars]

df_long = df_ny.melt(id_vars=id_vars, value_vars=date_cols, var_name='Date', value_name='Price')
df_long['Date'] = pd.to_datetime(df_long['Date'])
df_clean = df_long.dropna(subset=['Price']).sort_values(['RegionName','Date']).copy()

print("df_clean shape:", df_clean.shape)

df_clean shape: (8019, 7)


## 2. Feature Engineering


In [15]:
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month

df_model = df_clean.groupby('RegionName').apply(lambda g: g.assign(
    Price_Next_Month = g['Price'].shift(-1),
    Price_Lag_1M     = g['Price'].shift(1),
    Price_Lag_6M     = g['Price'].shift(6),
    Price_Lag_12M    = g['Price'].shift(12),
    Price_Roll_3M    = g['Price'].rolling(3, min_periods=1).mean().shift(1)
)).reset_index(drop=True)

df_model = df_model.dropna(subset=['Price_Next_Month', 'Price_Lag_12M'])
print("df_model ready:", df_model.shape)

df_model ready: (7681, 14)


/var/folders/lh/c7rm8s2j7bg03wh7jwz5q_sm0000gn/T/ipykernel_2325/1397014725.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_model = df_clean.groupby('RegionName').apply(lambda g: g.assign(


## 3. Exploratory Data Analysis

In [16]:
# 1. Average price over time
plt.figure(figsize=(14,6))
ny_avg = df_clean.groupby('Date')['Price'].mean()
plt.plot(ny_avg.index, ny_avg.values, linewidth=3, color='#1f77b4')
plt.title('Average Home Value in New York State (All Regions)', fontweight='bold')
plt.ylabel('Price ($)')
plt.xlabel('Year')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Top 10 most expensive regions (latest month)
latest_date = df_clean['Date'].max()
top10 = df_clean[df_clean['Date']==latest_date].nlargest(10, 'Price')
plt.figure(figsize=(12,7))
sns.barplot(data=top10, y='RegionName', x='Price', palette='viridis')
plt.title(f'Top 10 Most Expensive NY Regions — {latest_date.strftime("%B %Y")}', fontweight='bold')
plt.xlabel('Price ($)')
plt.tight_layout()
plt.show()

# 3. Price distribution
plt.figure(figsize=(11,5))
sns.histplot(df_clean['Price'], bins=60, kde=True, color='#d62728', alpha=0.9)
plt.title('Distribution of Home Prices in New York State', fontweight='bold')
plt.xlabel('Price ($)')
plt.axvline(df_clean['Price'].mean(), color='black', linestyle='--', linewidth=2,
            label=f"Mean = ${df_clean['Price'].mean():,.0f}")
plt.legend()
plt.tight_layout()
plt.show()

# 4. Correlation heatmap
corr_features = ['Price', 'Price_Lag_1M', 'Price_Lag_6M', 'Price_Lag_12M', 'Price_Roll_3M']
corr = df_model[corr_features].corr()

plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Price and Engineered Features', fontweight='bold')
plt.tight_layout()
plt.show()

/var/folders/lh/c7rm8s2j7bg03wh7jwz5q_sm0000gn/T/ipykernel_2325/59185353.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top10, y='RegionName', x='Price', palette='viridis')


## 4. Train/Test Split (Time-Series Aware)

In [18]:
features = ['Price_Lag_1M','Price_Lag_6M','Price_Lag_12M','Price_Roll_3M','Year','Month','SizeRank']
X = df_model[features]
y = df_model['Price_Next_Month']

# Time-series aware split
SPLIT_DATE = '2023-01-01'
train_mask = df_model['Date'] < SPLIT_DATE
test_mask  = df_model['Date'] >= SPLIT_DATE

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# Scale only the price-based lags
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[['Price_Lag_1M','Price_Lag_6M','Price_Lag_12M','Price_Roll_3M']])
X_test_scaled  = scaler.transform(X_test[['Price_Lag_1M','Price_Lag_6M','Price_Lag_12M','Price_Roll_3M']])

# Reattach non-scaled columns
X_train_final = np.hstack([X_train[['Year','Month','SizeRank']].values, X_train_scaled])
X_test_final  = np.hstack([X_test[['Year','Month','SizeRank']].values,  X_test_scaled])

## 5. Model Training & Hyperparameter Tuning

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_final, y_train)
print("Linear Regression R²:", r2_score(y_test, lr.predict(X_test_final)))

# Random Forest + RandomSearch
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators':[100,300], 'max_depth':[10,20,None], 'min_samples_split':[2,5]},
    n_iter=10, cv=3, scoring='neg_mean_squared_error', random_state=42, n_jobs=1)
rf_search.fit(X_train_final, y_train)
rf_best = rf_search.best_estimator_
print("RF Best Params:", rf_search.best_params_)
print("RF R²:", r2_score(y_test, rf_best.predict(X_test_final)))

# XGBoost + RandomSearch
xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', random_state=42),
    {'n_estimators':[100,300], 'max_depth':[3,5,7], 'learning_rate':[0.01,0.1], 'colsample_bytree':[0.7,1.0]},
    n_iter=10, cv=3, scoring='neg_mean_squared_error', random_state=42, n_jobs=1)
xgb_search.fit(X_train_final, y_train)
xgb_best = xgb_search.best_estimator_
print("XGBoost Best Params:", xgb_search.best_params_)
print("XGBoost R²:", r2_score(y_test, xgb_best.predict(X_test_final)))


Linear Regression R²: 0.9998420235297081
RF Best Params: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': None}
RF R²: 0.9905610960809177
XGBoost Best Params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
XGBoost R²: 0.9856720969979135

FINAL MODEL COMPARISON


,Model,R²,MAE
0,Linear Regression,0.9998,1121.5580
1,Random Forest,0.9906,4370.3264
2,XGBoost,0.9857,6130.2773


## 6. Final Model Comparison


In [ ]:

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'R²': [r2_score(y_test, lr.predict(X_test_final)),
           r2_score(y_test, rf_best.predict(X_test_final)),
           r2_score(y_test, xgb_best.predict(X_test_final))],
    'MAE': [mean_absolute_error(y_test, lr.predict(X_test_final)),
            mean_absolute_error(y_test, rf_best.predict(X_test_final)),
            mean_absolute_error(y_test, xgb_best.predict(X_test_final))]
})
print("\nFINAL MODEL COMPARISON")
display(results.round(4))